In [5]:
import pandas as pd
from pathlib import Path
import plotly.express as px
from dash import Dash, callback, html, dcc, Input, Output, register_page
import plotly.graph_objects as go
from prophet import Prophet
import gc

from data_handling import get_filtered_data

In [6]:
units = {'no': 'ppb',
 'no2': 'µg/m³',
 'o3': 'µg/m³',
 'co': 'µg/m³',
 'so2': 'µg/m³',
 'temperature': 'c',
 'pm25': 'µg/m³',
 'relativehumidity': '%',
 'um003': 'particles/cm³',
 'pm10': 'µg/m³',
 'pm1': 'µg/m³',
 'bc': 'µg/m³',
 'nox':'ppm',
 'no': 'ppm',
 'wind_speed': 'm/s',
 'wind_direction': 'degrees'}

In [7]:
pollutant_name = {'no': 'Nitric Oxide (NO)',
 'no2': 'Nitrogen Dioxide (NO2)',
 'o3': 'Ozone (O3)',
 'co': 'Carbon Monoxide (CO)',
 'so2': 'Sulfur Dioxide (SO2)',
 'temperature': 'Temperature',
 'pm25': 'Particulate Matter (PM2.5)',
 'relativehumidity': 'Relative Humidity',
 'um003': 'Ultrafine Particles (UM003)',
 'pm10': 'Particulate Matter (PM10)',
 'pm1': 'Particulate Matter (PM1)',
 'bc': 'Black Carbon (BC)',
 'nox':'Nitrogen Oxides (NOx)',
 'no': 'Nitric Oxide (NO)',
 'wind_speed': 'Wind Speed',
 'wind_direction': 'Wind Direction'}

In [8]:
city_names = ['Chicago', 'Sacramento', 'Bangalore', 'New Delhi']
for city_name in city_names:
    for pollutant in pollutant_name.keys():
        df = get_filtered_data(city_name)
        if pollutant in df.columns:
            df = df[pollutant].dropna().reset_index()
            df.rename({'Timestamp':'ds',pollutant: 'y'}, axis=1, inplace=True)
            # df['ds'] = df['ds'].dt.tz_localize(None)
            df['ds'] = pd.to_datetime(df['ds']).apply(lambda x: x.replace(tzinfo=None))
            # df['ds'] = pd.to_datetime(df['ds']).dt.tz_localize(None)
            model = Prophet(
            yearly_seasonality=True,
            weekly_seasonality=True, 
            daily_seasonality=True,
            interval_width=0.80,      # Reduced from 0.95
            uncertainty_samples=100,   # Reduced from 1000 (10x less memory!)
            n_changepoints=15,        # Reduced from default 25
            changepoint_prior_scale=0.05,
            mcmc_samples=0,            # Disable MCMC for faster processing
            stan_backend='CMDSTANPY'
            )
            model.fit(df)
            future = model.make_future_dataframe(periods=1500, freq='H')
            forecast = model.predict(future)
            
            trial_forecast = forecast[['ds','yhat','yhat_lower','yhat_upper']].copy()
            trial_forecast.to_parquet(f"./AQ_data_prediction/{city_name}_{pollutant}.gz", compression='gzip', index=False)
            del model
            print(f"Completed forecast for {city_name} - {pollutant}")
            gc.collect()

C:\Users\heman\AppData\Local\Temp\ipykernel_35728\2127839198.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['ds'] = pd.to_datetime(df['ds']).apply(lambda x: x.replace(tzinfo=None))
20:29:07 - cmdstanpy - INFO - Chain [1] start processing
20:29:15 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\heman\OneDrive\air quality digital twin hosting\.venv\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


Completed forecast for Chicago - o3


C:\Users\heman\AppData\Local\Temp\ipykernel_35728\2127839198.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['ds'] = pd.to_datetime(df['ds']).apply(lambda x: x.replace(tzinfo=None))
20:29:18 - cmdstanpy - INFO - Chain [1] start processing
20:29:25 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\heman\OneDrive\air quality digital twin hosting\.venv\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


Completed forecast for Chicago - temperature


C:\Users\heman\AppData\Local\Temp\ipykernel_35728\2127839198.py:9: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['ds'] = pd.to_datetime(df['ds']).apply(lambda x: x.replace(tzinfo=None))
20:29:32 - cmdstanpy - INFO - Chain [1] start processing
20:29:53 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\heman\OneDrive\air quality digital twin hosting\.venv\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


...

c:\Users\heman\OneDrive\air quality digital twin hosting\.venv\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


Completed forecast for New Delhi - wind_speed


20:39:38 - cmdstanpy - INFO - Chain [1] start processing
20:39:38 - cmdstanpy - INFO - Chain [1] done processing


Completed forecast for New Delhi - wind_direction


c:\Users\heman\OneDrive\air quality digital twin hosting\.venv\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


In [4]:
city_name = "Chicago"
pollutant = "temperature"
df = get_filtered_data(city_name)
df = df[pollutant].dropna().reset_index()
df.rename({'Timestamp':'ds',pollutant: 'y'}, axis=1, inplace=True)
# df['ds'] = df['ds'].dt.tz_localize(None)
df['ds'] = pd.to_datetime(df['ds']).apply(lambda x: x.replace(tzinfo=None))

C:\Users\heman\AppData\Local\Temp\ipykernel_15924\4238356973.py:7: FutureWarning: In a future version of pandas, parsing datetimes with mixed time zones will raise an error unless `utc=True`. Please specify `utc=True` to opt in to the new behaviour and silence this warning. To create a `Series` with mixed offsets and `object` dtype, please use `apply` and `datetime.datetime.strptime`
  df['ds'] = pd.to_datetime(df['ds']).apply(lambda x: x.replace(tzinfo=None))


In [5]:
model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True, 
    daily_seasonality=True,
    interval_width=0.80,      # Reduced from 0.95
    uncertainty_samples=100,   # Reduced from 1000 (10x less memory!)
    n_changepoints=15,        # Reduced from default 25
    changepoint_prior_scale=0.05,
    mcmc_samples=0,            # Disable MCMC for faster processing
    stan_backend='CMDSTANPY'
)

In [6]:
model.fit(df)

future = model.make_future_dataframe(periods=1500, freq='H')  # hourly frequency

19:53:02 - cmdstanpy - INFO - Chain [1] start processing
19:53:09 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\heman\OneDrive\air quality digital twin hosting\.venv\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  dates = pd.date_range(


In [7]:
# Predict
forecast = model.predict(future)

train_end = df['ds'].max()          # last date in the training set
future_only = forecast[forecast['ds'] > train_end]   # the *future* part
train_only  = forecast[forecast['ds'] <= train_end]  # the historical part
# train_only = train_only.resample('D', on='ds').mean().reset_index()

# Interactive forecast plot
fig = go.Figure()

# Historical
fig.add_trace(go.Scatter(x=df['ds'], y=df['y'],
                        mode='markers', name='Training points',
                        marker=dict(color='black')))

#historial forecast
fig.add_trace(go.Scatter(x=train_only['ds'], 
                            y=train_only['yhat'],
                            mode='lines', name='In-sample prediction',
                            line=dict(color='orange')))

# Future forecast
fig.add_trace(go.Scatter(x=future_only['ds'], y=future_only['yhat'],
                        mode='lines', name='Future forecast',
                        line=dict(color='steelblue')))

# Confidence band
fig.add_trace(go.Scatter(x=list(future_only['ds']) + list(future_only['ds'])[::-1],
                        y=list(future_only['yhat_upper']) + list(future_only['yhat_lower'])[::-1],
                        fill='toself', fillcolor='rgba(70, 130, 180, 0.2)',
                        line=dict(color='rgba(255,255,255,0)'),
                        name='confidence bounds',
                        hoverinfo='skip', showlegend=True))

fig.update_layout(title=dict(text=f"{city_name} - Trend and forecast of {pollutant_name[pollutant]}",
                        x=0.5,
                        font=dict(color='black', size=24)),
                xaxis_title='Date', yaxis_title=units[pollutant], width=1000, height=500)

fig.show()

In [8]:
trial_forecast = forecast[['ds','yhat','yhat_lower','yhat_upper']].copy()

In [9]:
trial_forecast.to_parquet(f"./AQ_data_prediction/{city_name}_{pollutant}.gz", compression='gzip', index=False)

In [10]:
trial_forecast = pd.read_parquet(f"./AQ_data_prediction/{city_name}_{pollutant}.gz")

In [12]:
forecast_2 = trial_forecast

train_end = df['ds'].max()          # last date in the training set
future_only = forecast_2[forecast_2['ds'] > train_end]   # the *future* part
train_only  = forecast_2[forecast_2['ds'] <= train_end]  # the historical part
# train_only = train_only.resample('D', on='ds').mean().reset_index()

# Interactive forecast plot
fig = go.Figure()

# Historical
fig.add_trace(go.Scatter(x=df['ds'], y=df['y'],
                        mode='markers', name='Training points',
                        marker=dict(color='black')))

#historial forecast
fig.add_trace(go.Scatter(x=train_only['ds'], 
                            y=train_only['yhat'],
                            mode='lines', name='In-sample prediction',
                            line=dict(color='orange')))

# Future forecast
fig.add_trace(go.Scatter(x=future_only['ds'], y=future_only['yhat'],
                        mode='lines', name='Future forecast',
                        line=dict(color='steelblue')))

# Confidence band
fig.add_trace(go.Scatter(x=list(future_only['ds']) + list(future_only['ds'])[::-1],
                        y=list(future_only['yhat_upper']) + list(future_only['yhat_lower'])[::-1],
                        fill='toself', fillcolor='rgba(70, 130, 180, 0.2)',
                        line=dict(color='rgba(255,255,255,0)'),
                        name='confidence bounds',
                        hoverinfo='skip', showlegend=True))

fig.update_layout(title=dict(text=f"{city_name} - Trend and forecast of {pollutant_name[pollutant]}",
                        x=0.5,
                        font=dict(color='black', size=24)),
                xaxis_title='Date', yaxis_title=units[pollutant], width=1000, height=500)

fig.show()